# Example queries: `helpers` (resstock_oedi)

Auto-generated from `tests/query_snapshots/helpers.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object


In [ ]:
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
)


## `upgrade_names_oedi`

get_upgrade_names — was broken before 2026-04-26 fix on two counts: (1) f-string interpolation of self.up_table (a Subquery) produced malformed `FROM SELECT * FROM ...` SQL with no enclosing parens; (2) the method assumed an `apply_upgrade.upgrade_name` column that doesn't exist on OEDI schemas. Fixed to use SA construction and gracefully degrade to `CAST(NULL AS VARCHAR) AS upgrade_name` when the name column is absent — so the dict shape stays consistent across schemas.


In [ ]:
result = bsq.get_upgrade_names()
result.head() if hasattr(result, 'head') else result


# shape: (16, 2)
 upgrade upgrade_name
       1          NaN
       2          NaN
       3          NaN
       4          NaN
       5          NaN


## `upgrades_csv_three_buildings_upgrade1`

get_upgrades_csv for upgrade=1 restricted to a tiny known building_id list. Mirrors results_csv_three_buildings on the upgrade-table side.


In [ ]:
result = bsq.get_upgrades_csv(upgrade_id='1', restrict=[('bldg_id', [1, 2, 3])])
result.head() if hasattr(result, 'head') else result


# shape: (3, 339)
 upgrade     weight  applicability  in.sqft  in.representative_income               in.ahs_region in.aiannh_area in.area_median_income in.ashrae_iecc_climate_zone_2004 in.ashrae_iecc_climate_zone_2004_2_a_split in.bathroom_spot_vent_hour in.battery in.bedrooms in.building_america_climate_zone in.cec_climate_zone      in.ceiling_fan in.census_division in.census_division_recs in.census_region               in.city in.clothes_dryer in.clothes_dryer_usage_level in.clothes_washer in.clothes_washer_presence in.clothes_washer_usage_level    in.cooking_range in.cooking_range_usage_level in.cooling_setpoint in.cooling_setpoint_has_offset in.cooling_setpoint_offset_magnitude in.cooling_setpoint_offset_period            in.corridor in.county  in.county_and_puma   in.county_name in.dehumidifier in.dishwasher in.dishwasher_usage_level in.door_area   in.doors in.duct_leakage_and_insulation in.duct_location in.eaves in.electric_vehicle                                                

## `buildings_by_locations_state`

get_buildings_by_locations: list bs-keys whose location_col=$STATE_COL_BASELINE is in ['CO']. Different code path from restrict — uses bs_key_cols projection and IN-list filter directly.


In [ ]:
result = bsq.get_buildings_by_locations(location_col='in.state', locations=['CO'])
result.head() if hasattr(result, 'head') else result


# shape: (9425, 1)
 bldg_id
      13
      16
      33
     128
     182


## `results_csv_three_buildings`

get_results_csv restricted to a tiny known building_id list. Pins the SELECT-* shape of the baseline metadata projection used by basic_usage.ipynb. Bounded to 3 buildings so the data parquet stays small (results_csv has ~150 columns, full state would be 100+ MB).


In [ ]:
result = bsq.get_results_csv(restrict=[('bldg_id', [1, 2, 3])])
result.head() if hasattr(result, 'head') else result


# shape: (3, 339)
 upgrade     weight  applicability  in.sqft  in.representative_income               in.ahs_region in.aiannh_area in.area_median_income in.ashrae_iecc_climate_zone_2004 in.ashrae_iecc_climate_zone_2004_2_a_split in.bathroom_spot_vent_hour in.battery in.bedrooms in.building_america_climate_zone in.cec_climate_zone      in.ceiling_fan in.census_division in.census_division_recs in.census_region               in.city in.clothes_dryer in.clothes_dryer_usage_level in.clothes_washer in.clothes_washer_presence in.clothes_washer_usage_level    in.cooking_range in.cooking_range_usage_level in.cooling_setpoint in.cooling_setpoint_has_offset in.cooling_setpoint_offset_magnitude in.cooling_setpoint_offset_period            in.corridor in.county  in.county_and_puma   in.county_name in.dehumidifier in.dishwasher in.dishwasher_usage_level in.door_area   in.doors in.duct_leakage_and_insulation in.duct_location in.eaves in.electric_vehicle                                                

## `distinct_vals_state_baseline`

get_distinct_vals on the baseline-table state column. Pins the simple SELECT DISTINCT shape — used by notebooks to enumerate categorical values before building restricts. The column name differs by schema (resstock=`in.state`, comstock=`state`); resolved via $STATE_COL_BASELINE.


In [ ]:
result = bsq.get_distinct_vals(column='in.state', table_name='resstock_2024_amy2018_release_2_metadata')
result.head() if hasattr(result, 'head') else result


# shape: (49, 1)
in.state
      CT
      CO
      WI
      MS
      ID


## `distinct_count_state_baseline`

get_distinct_count on the baseline-table state column. Pins the COUNT-with-weighted-sum shape (sample_count + weighted_count per category).


In [ ]:
result = bsq.get_distinct_count(column='in.state')
result.head() if hasattr(result, 'head') else result


# shape: (49, 3)
in.state  sample_count  weighted_count
      AL          9120    2.300991e+06
      AR          5536    1.396742e+06
      AZ         12023    3.033423e+06
      CA         57394    1.448060e+07
      CO          9425    2.377943e+06
